## 案例1

In [ ]:
import xarray as xr
file_name='D:/Onedrive/data/ERA5temp_1978_monthly.nc' 
ds = xr.open_dataset(file_name, engine='netcdf4')
ds.t2m[0].plot()

修改代码，提高绘图的美观度，添加投影，格式化经纬度，使用颜色卡，设置colorbar，添加全球陆地，经纬线等等

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# 读取数据
file_name='D:/Onedrive/data/ERA5temp_1978_monthly.nc' 
ds = xr.open_dataset(file_name, engine='netcdf4')

# 创建一个更美观的图形
plt.figure(figsize=(8, 6))

# 设置投影 - 使用Robinson投影
ax = plt.axes(projection=ccrs.Robinson())

# 添加全球陆地特征
ax.add_feature(cfeature.LAND, facecolor='lightgray', edgecolor='black', alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)

# 添加经纬网格线
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# 创建自定义颜色映射 - 从冷到热
colors = ["#2166ac", "#4393c3", "#92c5de", "#d1e5f0", 
          "#fddbc7", "#f4a582", "#d6604d", "#b2182b"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

# 绘制温度数据
temp_data = ds.t2m[0] - 273.15  # 转换为摄氏度
im = ax.pcolormesh(temp_data.longitude, temp_data.latitude, temp_data, 
                  transform=ccrs.PlateCarree(), cmap=cmap, 
                  vmin=-40, vmax=40)

# 添加色标
cbar = plt.colorbar(im, orientation='horizontal', pad=0.05, shrink=0.8)
cbar.set_label('Temperature (°C)')

# 添加标题
plt.title('Global Temperature Distribution', fontsize=16, pad=20)

# 设置地图范围
ax.set_global()

plt.tight_layout()
plt.show()

## 案例二

In [ ]:
import pandas as pd
data = pd.read_csv('machine_l.csv')
data.head()
# 数据预处理
# 将日期列转换为日期时间格式
data['Date'] = pd.to_datetime(data['Date'])
data = data.set_index('Date')


假设data包括以下列，写一段收敛交叉映射代码，模拟径流并进行径流归因：Date	LWSC	Runoff	P	E	Area	Runoff m3

In [ ]:
import pandas as pd
import pyEDM

# 读取数据
data = pd.read_csv('machine_l.csv')

# 数据预处理
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

# 提取需要分析的变量
variables = ['LWSC', 'Runoff', 'P', 'E', 'Area', 'Runoff m3']
data_ccm = data[variables].dropna()

# 收敛交叉映射（CCM）分析
# 例如，分析降水（P）和蒸散发（E）对径流（Runoff）的因果关系
lib_sizes = range(10, len(data_ccm)//2, 10)

# P对Runoff的因果关系
ccm_P_Runoff = pyEDM.CCM(dataFrame=data_ccm, E=3, Tp=0, columns='P', target='Runoff', libSizes=lib_sizes, sample=100)

# E对Runoff的因果关系
ccm_E_Runoff = pyEDM.CCM(dataFrame=data_ccm, E=3, Tp=0, columns='E', target='Runoff', libSizes=lib_sizes, sample=100)

# 输出CCM结果
print("CCM results for P causing Runoff:")
print(ccm_P_Runoff)

print("\nCCM results for E causing Runoff:")
print(ccm_E_Runoff)

## 案例三

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load and filter data
data = pd.read_excel('D:/Onedrive/acad1/data/flux/GPPdyn.xlsx')
data = data[data['GPPdyn'] > 0]

# Define plot variables
xlable, ylable = "GPP_DT", "GPPdyn"
x = data[xlable]
y = data[ylable]

# Set plot style
rc = {'font.sans-serif': ['Times New Roman']}
sns.set(context='notebook', style="darkgrid", font_scale=1, rc=rc)

# Create scatter plot with regression line
g = sns.lmplot(
    data=data,
    x=xlable, y=ylable, 
    hue="type",
    height=5
)

# Get the first subplot axis
ax = g.axes[0, 0]
